# Reasoning-trace trajectory analysis (question level)

**Type:** observational research analysis. No LLM inference, no new labels, no causal claims.

**Unit of analysis = the question** (`benchmark::question_id`). All eligible samples of a question are pooled over its
compute levels (the compute level is only kept as a label in the figures). Every pooled number is built as
*samples -> question (equal weight) -> pooled*, and every confidence interval comes from a bootstrap that resamples **questions**.

**Length** is measured in *words* of the thinking text (whitespace tokens after `<br>` -> space and zero-width characters removed).

**Three question groups, each with its own section, its own figures and its own analysis**

| Group | Rule (per question, eligible samples only) | What is compared inside the question |
|---|---|---|
| ALL-CORRECT | every sample is correct | long samples vs short samples |
| MIXED | at least one correct and one incorrect sample | incorrect samples vs correct samples |
| ALL-WRONG | every sample is incorrect | long samples vs short samples |

**Length significance.** A length difference only enters the long-vs-short calculations when the question's *standardised std*
(`std(words) / mean(words)` over its samples) is above the `LEN_STD_QUANTILE` (75 %) quantile of that value over all questions.

**Figures.** Section 6/7/8 *display* the trajectory figure of **`SHOW_FRAC` (75 %) of the questions of each group** (picked evenly across the length-spread order, so the shown set stays representative): one panel per compute level, each panel showing only the
samples of that level (the compute levels are compared to each other only through the panel layout). Only the overall figures and
`N_SAVE_PER_GROUP` example questions per group are written to disk, so the output folder stays small.

Sections: 1 Config -> 2 Load and data quality -> 3 Segment reconstruction -> 4 Eligibility -> 5 Features and question groups ->
6 ALL-CORRECT -> 7 MIXED -> 8 ALL-WRONG -> 9 Group comparison (5 conditions) -> 10 Saved files.

## 1. Config, imports and helpers

In [ ]:
import os, re, json, hashlib, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from IPython.display import display

# ------------------------------------------------------------------ paths
CSV_PATH = os.environ.get("TRACE_CSV", "../../data/processed/r1-llama8b/r1-llama8b_raw_results_segmented_labeled.csv")
OUT_DIR = os.environ.get("TRACE_OUT", "../../results/r1-llama8b/textual_analysis")

# ------------------------------------------------------------------ analysis parameters (defaults, no edits needed)
SEED = 0
MIN_SAMPLES = 5             # a question needs >= this many eligible samples (all compute levels together)
LEN_STD_QUANTILE = float(os.environ.get("TRACE_LEN_Q", 0.85))   # long-vs-short only for questions with std/mean above this quantile
SIDE_FRAC = 0.25            # "long" / "short" side = longest / shortest 25 % of a question's samples (at least 1 sample)
N_BOOT = 2000               # question-level bootstrap resamples
REL_BINS = 10               # relative-position bins (deciles)
FIG_DPI_SHOW = 75           # dpi of the figures shown in the notebook (small = light notebook)
CMP_LEN_SIG_ONLY = True     # section 9: ALL-CORRECT and ALL-WRONG enter the group comparison only with len_sig questions (same rule as 6.2 / 8.2)
N_SAVE_PER_GROUP = 5        # example questions saved to disk per group (the overall figures are always saved)
SHOW_FRAC = 0.75             # fraction of the questions of each group whose trajectory figure is DISPLAYED (keeps the notebook small)
MAX_SHOWN_PER_GROUP = None  # optional extra cap on the number of displayed figures per group (None = no cap)

ROLES = ["setup", "reasoning", "doubt", "verification", "correction"]
KEY_ROLES = ["doubt", "verification", "correction"]
STATUSES = ["none", "candidate", "final"]
TEMPORAL = ["new", "continuation", "repetition"]
# --- repetition arrows in the trajectory figures (phase level only)
REP_VALUES = {"repetition", "repeat", "repeated", "repetitive"}   # temporal_role values that mean "repeats earlier content"
REP_SAME_ROLE = False       # True = the arrow may only point to an earlier phase with the same discourse_role
REP_COLOR = "#222222"
NEW_VALUES = {"new"}        # temporal_role values that start a NEW thought
SEP_COLOR = "#000000"       # black bar drawn at the start of EVERY `new` segment (same phase or not)
REQUIRED_COLS = ["question_id", "benchmark", "compute_level", "sample_index", "thinking_text", "gold", "is_correct",
                 "was_truncated", "rtl_labels_json"]
OPTIONAL_COLS = ["difficulty_tier", "task_type", "rtl_n_segments"]      # used when present
GROUPS = ["ALL-CORRECT", "MIXED", "ALL-WRONG"]

FIG_DIR = os.path.join(OUT_DIR, "figures")
EX_DIR = os.path.join(FIG_DIR, "examples")
TAB_DIR = os.path.join(OUT_DIR, "tables")
for d in [FIG_DIR, EX_DIR, TAB_DIR]:
    os.makedirs(d, exist_ok=True)

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 80)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice")     # expected: bins / questions that no trace reaches
warnings.filterwarnings("ignore", message="All-NaN slice")

# One colour per discourse role, identical in every figure.
ROLE_COLORS = {"setup": "#9aa5b1", "reasoning": "#4c78a8", "doubt": "#f58518", "verification": "#54a24b",
               "correction": "#e45756"}
COND_COLORS = {"ALL-CORRECT short": "#9ecae1", "ALL-CORRECT long": "#08306b", "MIXED correct": "#74c476",
               "MIXED incorrect": "#f58518", "ALL-WRONG": "#d62728"}

ZW_RE = re.compile("[\u200b\u200c\u200d\u2060\ufeff]")
BR_RE = re.compile(r"<br\s*/?>", re.I)
MARK_RE = re.compile("\u27e6S(\\d+)\u27e7")


# ------------------------------------------------------------------ text helpers
def clean_text(s):
    """<br> -> space, zero-width characters removed. Used for word counts and text analyses only."""
    return ZW_RE.sub("", BR_RE.sub(" ", s))


def word_count(s):
    return len(clean_text(s).split())


def norm_answer(x):
    """Light answer normalisation (case, \\boxed{}, $, spaces, thousands commas, trailing period; numbers canonical)."""
    if x is None or (not isinstance(x, str) and pd.isna(x)):
        return None
    s = str(x).strip().lower()
    s = re.sub(r"\\boxed\s*\{(.*)\}", r"\1", s)
    s = re.sub(r"\\text\s*\{(.*?)\}", r"\1", s)
    s = s.replace("\\dfrac", "\\frac").replace("\\left", "").replace("\\right", "")
    s = re.sub(r"\\[,!; ]", "", s).replace("$", "").replace(" ", "")
    s = re.sub(r"(?<=\d),(?=\d{3}(\D|$))", "", s)
    s = s.rstrip(".")
    if s == "":
        return None
    try:
        return "%.10g" % float(s)
    except ValueError:
        return s


def is_refinement(a, b):
    """Two different normalised answers where one contains the other (e.g. 'x=3' vs '3'): refinement / format change."""
    return a != b and (a in b or b in a)


def safe_name(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(s)).strip("-")[:80]


# ------------------------------------------------------------------ statistics helpers (question level)
EFF_COLS = ["feature", "n_questions", "mean", "ci_lo", "ci_hi", "dz", "dz_lo", "dz_hi", "p_signflip", "p_wilcoxon"]


def nanpct(x, q):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return np.percentile(x, q) if len(x) else np.nan


def paired_summary(qdf, cols=None, B=None, seed=SEED):
    """Per-question differences -> mean, bootstrap CI, dz (+CI), sign-flip p, Wilcoxon p. One row per column of qdf."""
    rng = np.random.default_rng(seed)
    B = B or N_BOOT
    cols = list(qdf.columns) if cols is None else cols
    rows = []
    for c in cols:
        v = qdf[c].dropna().to_numpy(float)
        n = len(v)
        if n < 3:
            rows.append(dict(feature=c, n_questions=n))
            continue
        vi = v[rng.integers(0, n, (B, n))]
        bm, bsd, sd = vi.mean(1), vi.std(1, ddof=1), v.std(ddof=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            bdz = np.where(bsd > 0, bm / bsd, np.nan)
        dz = v.mean() / sd if sd > 0 else np.nan
        flips = rng.choice([-1.0, 1.0], (B, n))
        p_sf = (1 + (np.abs((flips * v).mean(1)) >= abs(v.mean()) - 1e-12).sum()) / (B + 1)
        try:
            p_w = np.nan if np.all(v == 0) else stats.wilcoxon(v).pvalue
        except ValueError:
            p_w = np.nan
        rows.append(dict(feature=c, n_questions=n, mean=v.mean(), ci_lo=np.percentile(bm, 2.5), ci_hi=np.percentile(bm, 97.5),
                         dz=dz, dz_lo=nanpct(bdz, 2.5), dz_hi=nanpct(bdz, 97.5), p_signflip=p_sf, p_wilcoxon=p_w))
    return pd.DataFrame(rows, columns=EFF_COLS).set_index("feature")


def boot_mean_ci(v, B=None, seed=SEED):
    v = np.asarray(v, float)
    v = v[~np.isnan(v)]
    if len(v) < 3:
        return np.nan, np.nan, np.nan, len(v)
    rng = np.random.default_rng(seed)
    bm = v[rng.integers(0, len(v), (B or N_BOOT, len(v)))].mean(1)
    return v.mean(), np.percentile(bm, 2.5), np.percentile(bm, 97.5), len(v)


def boot_matrix_ci(X, B=300, seed=SEED):
    """Column-wise mean and bootstrap CI over the ROWS (questions) of a 2-D array that may contain NaN."""
    X = np.asarray(X, float)
    n = X.shape[0]
    if n == 0:
        nan = np.full(X.shape[1], np.nan)
        return nan, nan, nan
    rng = np.random.default_rng(seed)
    boots = np.empty((B, X.shape[1]))
    for b in range(B):
        boots[b] = np.nanmean(X[rng.integers(0, n, n)], axis=0)
    return np.nanmean(X, axis=0), np.nanpercentile(boots, 2.5, axis=0), np.nanpercentile(boots, 97.5, axis=0)


# ------------------------------------------------------------------ output helpers: save only what is asked for
def save_table(df, name, index=True):
    path = os.path.join(TAB_DIR, name)
    df.to_csv(path, index=index)
    return path


def finish_fig(fig, path=None, show=True):
    """Save ONLY when a path is given, show inline when asked, always close (no memory build-up)."""
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        fig.savefig(path, dpi=130, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)

## 2. Load and data quality
The CSV is read as text (so ids, answers and booleans survive untouched). Only the empty string is treated as missing;
`'None'`/`'NA'` stay as they are because they can be legitimate gold answers. Source columns are never renamed.

In [ ]:
raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=[""], encoding="utf-8")
missing_cols = [c for c in REQUIRED_COLS if c not in raw.columns]
assert not missing_cols, "Missing required columns: %s" % missing_cols
for c in OPTIONAL_COLS:
    if c not in raw.columns:
        raw[c] = np.nan
print("rows: %d | columns: %d | extra (unused) columns: %d" % (len(raw), raw.shape[1], raw.shape[1] - len(REQUIRED_COLS) - len(OPTIONAL_COLS)))


def parse_bool(s):
    m = {"true": 1.0, "1": 1.0, "1.0": 1.0, "yes": 1.0, "y": 1.0, "t": 1.0,
         "false": 0.0, "0": 0.0, "0.0": 0.0, "no": 0.0, "n": 0.0, "f": 0.0}
    return s.str.strip().str.lower().map(m)


df = raw.copy()
df["trace_id"] = np.arange(len(df))
df["qkey"] = df["benchmark"].astype(str) + "::" + df["question_id"].astype(str)
df["correct"] = parse_bool(df["is_correct"])             # 1.0 / 0.0 / NaN (unknown)
df["trunc"] = parse_bool(df["was_truncated"])            # 1.0 / 0.0 / NaN (unknown)
df["sample_idx"] = pd.to_numeric(df["sample_index"], errors="coerce")
print("values that could not be parsed as booleans: is_correct=%d, was_truncated=%d" % (
    df["correct"].isna().sum() - df["is_correct"].isna().sum(), df["trunc"].isna().sum() - df["was_truncated"].isna().sum()))
for c in ["benchmark", "compute_level"]:
    print("\n" + c); print(df[c].value_counts(dropna=False).to_string())
print("\nis_correct:", df["correct"].value_counts(dropna=False).to_dict(), "| was_truncated:", df["trunc"].value_counts(dropna=False).to_dict())

# ---- data quality (row level, before reconstruction)
KEY_COLS = ["benchmark", "question_id", "compute_level", "sample_index"]
df["dup_key"] = df.duplicated(KEY_COLS, keep=False)
has_text = df["thinking_text"].notna()
tt = df["thinking_text"].fillna("")
n_open, n_mark = tt.str.count("\u27e6"), tt.str.count(MARK_RE)
dq = pd.DataFrame([
    ("rows total", len(df), ""),
    ("duplicate (benchmark, question_id, compute_level, sample_index) key", df["dup_key"].sum(), "all rows sharing a duplicated key are excluded (ambiguous)"),
    ("empty thinking_text", (~has_text).sum(), ""),
    ("empty rtl_labels_json", df["rtl_labels_json"].isna().sum(), ""),
    ("correctness unknown", df["correct"].isna().sum(), ""),
    ("truncation status unknown", df["trunc"].isna().sum(), ""),
    ("gold missing", df["gold"].isna().sum(), ""),
    ("thinking_text contains <br> tags", tt.str.contains(BR_RE).sum(), "replaced by a space for counting"),
    ("thinking_text contains zero-width characters", tt.str.contains(ZW_RE).sum(), "removed for counting"),
    ("malformed markers (opening bracket count != valid marker count)", (n_open != n_mark).sum(), ""),
    ("no marker at all", ((n_mark == 0) & has_text).sum(), ""),
    ("same thinking_text twice inside one question", df[has_text].duplicated(["qkey", "thinking_text"], keep=False).sum(),
     "the same trace stored under several compute levels? only the first copy is kept (section 4)"),
], columns=["check", "n_rows", "note"])
dq["share"] = dq["n_rows"] / len(df)
display(dq)
print("\nsamples per question (all compute levels, before any exclusion):")
print(df.groupby("qkey").size().describe().round(1).to_dict())

In [ ]:
dup_traces = df[
    df["thinking_text"].notna()
    & df["thinking_text"].astype(str).str.strip().ne("")
    & df.duplicated(["qkey", "thinking_text"], keep=False)
].sort_values(["qkey", "compute_level", "sample_idx"])

display(dup_traces)

## 3. Segment reconstruction and annotation matching
Rules: `thinking_text` is split on the `⟦S<n>⟧` markers (the text after a marker belongs to that segment, original order kept).
Annotations are matched to segments **by `segment_id`**, never by position. A trace counts as *labeled* only if the marker ids in the
text and the annotation ids are the same set, there are no duplicate ids, the JSON is valid and **every segment has a valid
`discourse_role`**. Invalid `answer_status` / `temporal_role` values are set to missing (never guessed). The length of a trace is the sum of
its segment word counts (text before the first marker is reported separately).

In [ ]:
def parse_seg_num(seg_id):
    m = re.search(r"(\d+)", str(seg_id))
    return int(m.group(1)) if m else None


seg_rows, trace_flags = [], []
n_rtl_all = pd.to_numeric(df["rtl_n_segments"], errors="coerce").to_numpy()
for tid, text, lj, n_rtl in zip(df["trace_id"], df["thinking_text"], df["rtl_labels_json"], n_rtl_all):
    has_t, has_j = isinstance(text, str), isinstance(lj, str)
    fl = dict(trace_id=tid, no_text=not has_t, json_missing=not has_j, json_malformed=False, n_text_seg=0, n_json=0,
              dup_ids_text=False, dup_ids_json=False, ids_mismatch=False, count_mismatch=False, rtl_n_vs_text=False,
              bad_role=0, bad_status=0, bad_temporal=0, empty_segments=0, preamble_words=0)
    ann_by_num, ann = {}, []
    if has_j:
        try:
            ann = json.loads(lj)
            if not isinstance(ann, list) or not all(isinstance(a, dict) for a in ann):
                fl["json_malformed"], ann = True, []
        except json.JSONDecodeError:
            fl["json_malformed"], ann = True, []
        for a in ann:
            n = parse_seg_num(a.get("segment_id"))
            if n is None:
                continue
            if n in ann_by_num:
                fl["dup_ids_json"] = True
            else:
                ann_by_num[n] = a
        fl["n_json"] = len(ann)
    if has_t:
        ms = list(MARK_RE.finditer(text))
        fl["n_text_seg"] = len(ms)
        text_nums = [int(m.group(1)) for m in ms]
        fl["dup_ids_text"] = len(set(text_nums)) != len(text_nums)
        if ms:
            fl["preamble_words"] = word_count(text[:ms[0].start()])
        for i, m in enumerate(ms):
            end = ms[i + 1].start() if i + 1 < len(ms) else len(text)
            raw_seg = text[m.end():end]
            a = ann_by_num.get(int(m.group(1)))
            role = status = value = temporal = None
            if a is not None:
                role, status, temporal = a.get("discourse_role"), a.get("answer_status"), a.get("temporal_role")
                value = a.get("answer_value")
                if role not in ROLES:
                    fl["bad_role"] += 1
                if status not in STATUSES:
                    fl["bad_status"] += 1
                    status = None
                if temporal not in TEMPORAL:
                    fl["bad_temporal"] += 1
                    temporal = None
                if value is not None and not isinstance(value, str):
                    value = str(value)
            clean = clean_text(raw_seg)
            w = len(clean.split())
            fl["empty_segments"] += int(w == 0)
            seg_rows.append((tid, i, int(m.group(1)), raw_seg, clean, w, role, status, value, temporal))
        fl["ids_mismatch"] = set(text_nums) != set(ann_by_num)
        fl["count_mismatch"] = fl["n_text_seg"] != fl["n_json"]
        if n_rtl == n_rtl:
            fl["rtl_n_vs_text"] = int(n_rtl) != fl["n_text_seg"]
    trace_flags.append(fl)

seg_all = pd.DataFrame(seg_rows, columns=["trace_id", "seg_pos", "seg_num", "text_raw", "text_clean", "words", "role",
                                          "answer_status", "answer_value", "temporal_role"])
flags = pd.DataFrame(trace_flags).set_index("trace_id")
flags["labels_ok"] = (~flags["no_text"] & ~flags["json_missing"] & ~flags["json_malformed"] & (flags["n_text_seg"] > 0)
                      & ~flags["ids_mismatch"] & ~flags["dup_ids_text"] & ~flags["dup_ids_json"] & (flags["bad_role"] == 0))
# cumulative word positions in TEXT ORDER (never sorted by id)
seg_all["end_word"] = seg_all.groupby("trace_id")["words"].cumsum()
seg_all["start_word"] = seg_all["end_word"] - seg_all["words"]
flags["words_total"] = seg_all.groupby("trace_id")["words"].sum().reindex(flags.index).fillna(0).astype(int)

# Independent check: words of the whole text (markers replaced by spaces) must equal the sum of the segment words.
chk = df.set_index("trace_id")["thinking_text"].dropna()
first_mark = chk.map(lambda s: (MARK_RE.search(s).start() if MARK_RE.search(s) else len(s)))
indep = pd.Series({t: word_count(MARK_RE.sub(" ", s[first_mark[t]:])) for t, s in chk.items()})
assert (indep.reindex(flags.index).fillna(0).astype(int) == flags["words_total"]).all(), "segment widths do not sum to the trace length"
assert (seg_all.groupby("trace_id")["seg_pos"].apply(lambda s: (s.to_numpy() == np.arange(len(s))).all())).all()
assert (seg_all.groupby("trace_id")["start_word"].apply(lambda s: (np.diff(s.to_numpy()) >= 0).all())).all()
print("assertions passed: segment widths sum to trace length; text order preserved (%d traces, %d segments)" % (len(flags), len(seg_all)))

n = len(flags)
recon_report = pd.DataFrame([
    ("JSON missing / empty", flags["json_missing"].sum()),
    ("JSON malformed", flags["json_malformed"].sum()),
    ("segment count in text != annotation count", flags["count_mismatch"].sum()),
    ("segment-id set in text != annotation id set", flags["ids_mismatch"].sum()),
    ("duplicate segment ids (text)", flags["dup_ids_text"].sum()),
    ("duplicate segment ids (JSON)", flags["dup_ids_json"].sum()),
    ("rtl_n_segments != segments in text", flags["rtl_n_vs_text"].sum()),
    ("unexpected discourse_role", (flags["bad_role"] > 0).sum()),
    ("unexpected answer_status (set to missing)", (flags["bad_status"] > 0).sum()),
    ("unexpected temporal_role (set to missing)", (flags["bad_temporal"] > 0).sum()),
    ("with >= 1 empty (zero-word) segment", (flags["empty_segments"] > 0).sum()),
    ("with words before the first marker (excluded from length)", (flags["preamble_words"] > 0).sum()),
    ("LABELED traces (all checks passed)", flags["labels_ok"].sum()),
], columns=["check", "n_traces"])
recon_report["share"] = recon_report["n_traces"] / n
display(recon_report)

## 4. Eligibility funnel
A sample is **eligible** if its key is not duplicated, it is labeled, has words, has known correctness, is **complete (not truncated)**,
and its thinking text is the first copy of that text inside its question (the same trace stored under several compute levels would be
counted twice once the compute levels are pooled). Every exclusion is counted.

In [ ]:
tr = df.set_index("trace_id").join(flags)
tr["text_hash"] = tr["thinking_text"].map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest() if isinstance(s, str) else None)


def first_copy_in_question(alive):
    dup = tr[alive].duplicated(["qkey", "text_hash"], keep="first")
    return ~tr.index.isin(dup.index[dup])


steps = [
    ("not a duplicate key", lambda a: ~tr["dup_key"]),
    ("labeled (valid JSON, ids match, all roles valid)", lambda a: tr["labels_ok"]),
    ("has words", lambda a: tr["words_total"] > 0),
    ("correctness known", lambda a: tr["correct"].notna()),
    ("truncation status known", lambda a: tr["trunc"].notna()),
    ("complete (not truncated)", lambda a: tr["trunc"] == 0),
    ("first copy of its thinking text inside the question", first_copy_in_question),
]
alive = pd.Series(True, index=tr.index)
funnel = [dict(step="all rows", removed=0, remaining=len(tr))]
for name, fn in steps:
    ok = pd.Series(np.asarray(fn(alive)), index=tr.index)
    funnel.append(dict(step=name, removed=int((alive & ~ok).sum()), remaining=int((alive & ok).sum())))
    alive &= ok
funnel = pd.DataFrame(funnel)
display(funnel)
tr["eligible"] = alive
print("eligible samples: %d | questions with >= 1 eligible sample: %d" % (alive.sum(), tr.loc[alive, "qkey"].nunique()))

## 5. Features and question groups
Features are computed for **eligible samples only**. *Length-free* features (shares, rates per 100 words, relative positions) do not
change just because a trace is longer; length-dependent ones are listed separately. `final` is an annotation status, **not** correctness.
Text features are *cues and similarity scores*, not labels.

In [ ]:
BASE = ["benchmark", "question_id", "qkey", "compute_level", "sample_idx", "difficulty_tier", "task_type", "correct", "gold",
        "words_total"]
E = tr.loc[tr["eligible"], BASE].sort_index().rename(columns={"words_total": "words"}).copy()
S = seg_all[seg_all["trace_id"].isin(E.index)].sort_values(["trace_id", "seg_pos"]).reset_index(drop=True)
assert np.array_equal(S["trace_id"].unique(), E.index.to_numpy()), "every eligible trace needs >= 1 segment"
nE = len(E)
E["n_seg"] = S.groupby("trace_id").size()
E["log_words"] = np.log(E["words"])
E["mean_seg_words"] = E["words"] / E["n_seg"]
S["twords"] = S["trace_id"].map(E["words"])
S["rel_start"], S["rel_end"] = S["start_word"] / S["twords"], S["end_word"] / S["twords"]
S["role_code"] = S["role"].map({r: i for i, r in enumerate(ROLES)})
offs = np.r_[0, np.cumsum(S.groupby("trace_id", sort=True).size().to_numpy())]      # contiguous slice per trace

# ---- role composition (word shares) -------------------------------------------------------------------------------
role_words = S.groupby(["trace_id", "role"])["words"].sum().unstack(fill_value=0).reindex(columns=ROLES, fill_value=0)
for r in ROLES:
    E["share_" + r] = role_words[r].reindex(E.index).fillna(0) / E["words"]

# ---- merged phase runs (consecutive segments with the same role) --------------------------------------------------
S["run_break"] = S["role"].ne(S.groupby("trace_id")["role"].shift())
S["run_idx"] = S.groupby("trace_id")["run_break"].cumsum() - 1
R = S.groupby(["trace_id", "run_idx"], sort=True).agg(
    role=("role", "first"), words=("words", "sum"), start_word=("start_word", "min"), end_word=("end_word", "max"),
    first_temporal=("temporal_role", "first"), text=("text_clean", " ".join)).reset_index()
R["twords"] = R["trace_id"].map(E["words"])
R["rel_start"] = R["start_word"] / R["twords"]
R["prev_role"] = R.groupby("trace_id")["role"].shift(1)
R["next_role"] = R.groupby("trace_id")["role"].shift(-1)
E["n_runs"] = R.groupby("trace_id").size()
E["changes_per100w"] = (E["n_runs"] - 1) / E["words"] * 100
for r in KEY_ROLES:
    sel = R[R["role"] == r].groupby("trace_id")
    E["runs100_" + r] = sel.size().reindex(E.index).fillna(0) / E["words"] * 100
    E["first_%s_rel" % r] = sel["rel_start"].min().reindex(E.index)          # NaN when the trace never enters the role
for a, b, nm in [("doubt", "verification", "doubt_to_verif"), ("doubt", "correction", "doubt_to_corr"),
                 ("verification", "doubt", "verif_to_doubt")]:
    cnt = R[(R["role"] == a) & (R["next_role"] == b)].groupby("trace_id").size()
    E[nm + "100"] = cnt.reindex(E.index).fillna(0) / E["words"] * 100

# ---- hedge cues (surface phrases per 100 words; cues, not labels) ---------------------------------------------------
HEDGE = (r"\bwait\b|\bhmm+\b|\bmaybe\b|\bactually\b|\blet me (?:check|double[- ]?check|verify|re-?check)\b|\bdouble[- ]?check"
         r"|\balternatively\b|\bor perhaps\b|\bon the other hand\b")
S["cue_hedge"] = S["text_clean"].str.lower().str.count(HEDGE)
E["hedge_100w"] = S.groupby("trace_id")["cue_hedge"].sum().reindex(E.index) / E["words"] * 100

# ---- novelty and repetition (per trace, in text order) ---------------------------------------------------------------
TOK_RE = re.compile(r"[a-z0-9]+(?:[.'\-][a-z0-9]+)*")
SENT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
cols = ["n_content", "n_new", "n4", "n4_rep", "n_sent", "n_sent_rep"]
tx = {c: np.zeros(len(S)) for c in cols}
texts = S["text_clean"].to_numpy()
for ti in range(nE):
    a, b = offs[ti], offs[ti + 1]
    seen_c, seen4, seen_s = set(), set(), set()
    for k in range(a, b):
        t = texts[k].lower()
        toks = TOK_RE.findall(t)
        content = {w for w in toks if w.isalpha() and len(w) >= 3 and w not in ENGLISH_STOP_WORDS}
        tx["n_content"][k], tx["n_new"][k] = len(content), len(content - seen_c)
        seen_c |= content
        for i in range(len(toks) - 3):
            g = " ".join(toks[i:i + 4])
            tx["n4"][k] += 1
            if g in seen4:
                tx["n4_rep"][k] += 1
            else:
                seen4.add(g)
        for sent in SENT_RE.split(t):
            st = TOK_RE.findall(sent)
            if len(st) < 4:
                continue
            key = " ".join(st)
            tx["n_sent"][k] += 1
            if key in seen_s:
                tx["n_sent_rep"][k] += 1
            else:
                seen_s.add(key)
for c in cols:
    S[c] = tx[c]
tsum = S.groupby("trace_id")[cols].sum().reindex(E.index)
E["novelty"] = tsum["n_new"] / tsum["n_content"].replace(0, np.nan)
E["rep4_share"] = tsum["n4_rep"] / tsum["n4"].replace(0, np.nan)
E["sent_repeat_share"] = tsum["n_sent_rep"] / tsum["n_sent"].replace(0, np.nan)

# ---- repetition links between PHASES (merged runs) -------------------------------------------------------------------
# Rule: a phase (merged run) that is NOT the first one and whose FIRST segment has temporal_role = repetition gets a dashed arrow
# back to the earlier phase whose text is most similar to it (word overlap of content words, Jaccard).
# reasoning -> reasoning is ONE phase, so it never produces an arrow.
def _tokset(t):
    return {w for w in TOK_RE.findall(t.lower()) if len(w) >= 3 and w not in ENGLISH_STOP_WORDS}


REP_LINKS = {}      # trace_id -> [(word position where the repeating phase starts, word position in the middle of the source phase)]
n_rep_starts = n_no_src = 0
for tid, g in R.groupby("trace_id", sort=False):
    role_, st_, en_ = g["role"].to_numpy(), g["start_word"].to_numpy(), g["end_word"].to_numpy()
    is_rep_ = g["first_temporal"].isin(REP_VALUES).to_numpy()
    if not is_rep_[1:].any():
        continue
    ts_ = [_tokset(t) for t in g["text"]]
    links = []
    for j in range(1, len(g)):
        if not is_rep_[j]:
            continue
        n_rep_starts += 1
        cand = [i for i in range(j) if (not REP_SAME_ROLE) or role_[i] == role_[j]]
        sim = [len(ts_[j] & ts_[i]) / max(1, len(ts_[j] | ts_[i])) for i in cand]
        if not cand or max(sim) == 0:
            n_no_src += 1
            continue
        i = cand[int(np.argmax(sim))]
        links.append((st_[j], (st_[i] + en_[i]) / 2))
    if links:
        REP_LINKS[tid] = links
print("phases that start with a repetition label: %d | arrows drawn: %d | no similar earlier phase found: %d" % (
    n_rep_starts, sum(len(v) for v in REP_LINKS.values()), n_no_src))

# ---- answer development ------------------------------------------------------------------------------------------------
# Positions of answer events are measured at the END of the segment that first carries them ("the answer has been stated by here").
S["ans_norm"] = S["answer_value"].map(norm_answer)
E["gold_norm"] = E["gold"].map(norm_answer)
isval = lambda v: isinstance(v, str)           # a real normalised value (None / NaN are both "no value")
ev = S[S["answer_status"].isin(["candidate", "final"]) | S["ans_norm"].notna()]
ans_rows = []
for tid, g in ev.groupby("trace_id", sort=False):
    pos, status, val = g["seg_pos"].to_numpy(), g["answer_status"].to_numpy(), g["ans_norm"].to_numpy()
    endw = dict(zip(pos, g["end_word"].to_numpy()))
    is_c, is_f = status == "candidate", status == "final"
    first_c = pos[is_c][0] if is_c.any() else None
    first_f = pos[is_f][0] if is_f.any() else None
    first_a = pos[is_c | is_f][0] if (is_c | is_f).any() else None
    vals = [(p, v) for p, v in zip(pos, val) if isval(v)]
    repl, refine, prev = [], [], None
    for p, v in vals:
        if prev is not None and v != prev:
            (refine if is_refinement(prev, v) else repl).append(p)
        prev = v
    last_val = vals[-1][1] if vals else None
    stab = None
    if vals:
        i = len(vals) - 1
        while i > 0 and vals[i - 1][1] == last_val:
            i -= 1
        stab = vals[i][0]
    gold_n = E.at[tid, "gold_norm"]
    gold_hit = [p for p, v in vals if isval(gold_n) and v == gold_n]
    last_ok = float(last_val is not None and isval(gold_n) and last_val == gold_n)
    ans_rows.append(dict(trace_id=tid, first_ans_seg=first_a, first_cand_seg=first_c, first_final_seg=first_f,
                         first_ans_end=endw.get(first_a), first_cand_end=endw.get(first_c), first_final_end=endw.get(first_f),
                         n_replace=len(repl), n_refine=len(refine), replace_segs=repl, refine_segs=refine, last_val=last_val,
                         stab_end=endw[stab] if stab is not None else np.nan,
                         gold_ever=float(bool(gold_hit)), last_matches_gold=last_ok,
                         gold_lost=float(bool(gold_hit) and not last_ok)))
A = pd.DataFrame(ans_rows).set_index("trace_id")
A_lists = A[["replace_segs", "refine_segs"]]               # positions of answer-change events (used by the trajectory figures)
E = E.join(A.drop(columns=["replace_segs", "refine_segs"]), how="left")
E["has_answer"] = E["first_ans_seg"].notna().astype(float)
E["first_ans_rel"] = E["first_ans_end"] / E["words"]
E["first_final_rel"] = E["first_final_end"] / E["words"]
E["stab_rel"] = E["stab_end"] / E["words"]
E["share_after_first_ans"] = 1 - E["first_ans_rel"]
E["share_after_stab"] = 1 - E["stab_rel"]
E["n_replace"], E["n_refine"] = E["n_replace"].fillna(0), E["n_refine"].fillna(0)
E["n_replace_100w"] = E["n_replace"] / E["words"] * 100
E["any_replace"] = (E["n_replace"] > 0).astype(float)
no_val = E["last_val"].isna()          # no annotated answer value -> gold comparisons are undefined, not "0"
for c in ["gold_ever", "gold_lost", "last_matches_gold"]:
    E.loc[no_val, c] = np.nan
E["ans_pattern"] = np.select([no_val, E["gold_lost"] == 1, E["n_replace"] > 0],
                             ["no answer value", "gold candidate later replaced", "answer replaced"], default="stable")
PAT_LABELS = {"pat_stable": "stable", "pat_replaced": "answer replaced",
              "pat_goldlost": "gold candidate later replaced (outcome-linked)", "pat_noanswer": "no answer value"}
for col, lab in zip(PAT_LABELS, ["stable", "answer replaced", "gold candidate later replaced", "no answer value"]):
    E[col] = (E["ans_pattern"] == lab).astype(float)

# ---- feature sets --------------------------------------------------------------------------------------------------------
LEN_FREE = (["share_" + r for r in ROLES] + ["changes_per100w"] + ["runs100_" + r for r in KEY_ROLES]
            + ["doubt_to_verif100", "doubt_to_corr100", "verif_to_doubt100"] + ["first_%s_rel" % r for r in KEY_ROLES]
            + ["first_ans_rel", "first_final_rel", "stab_rel", "share_after_first_ans", "share_after_stab", "n_replace_100w", "any_replace"]
            + ["hedge_100w", "novelty", "rep4_share", "sent_repeat_share"])
LEN_DEP = ["words", "log_words", "n_seg", "mean_seg_words"]
PAT_COLS = list(PAT_LABELS)
TABLE_FEATS = LEN_FREE + LEN_DEP + PAT_COLS

# ---- role composition inside relative-position deciles (used by every profile figure) --------------------------------
S_start, S_words, S_end = S["start_word"].to_numpy(), S["words"].to_numpy(), S["end_word"].to_numpy()
S_role = S["role"].to_numpy()
S_temp = S["temporal_role"].to_numpy(dtype=object)
code = E.index.get_indexer(S["trace_id"]) * 5 + S["role_code"].to_numpy()
tw_seg = S["twords"].to_numpy(float)
REL = np.zeros((nE, 5, REL_BINS))
for b in range(REL_BINS):
    lo, hi = b / REL_BINS * tw_seg, (b + 1) / REL_BINS * tw_seg
    ov = np.clip(np.minimum(S_end, hi) - np.maximum(S_start, lo), 0, None)
    REL[:, :, b] = np.bincount(code, weights=ov, minlength=nE * 5).reshape(nE, 5)
assert np.allclose(REL.sum(axis=(1, 2)), E["words"].to_numpy(float)), "relative bins must add up to the trace length"
REL_FLAT = (REL / (E["words"].to_numpy(float)[:, None, None] / REL_BINS)).reshape(nE, -1)    # (trace, role*bin): role share in each decile
seg_ix = S.groupby("trace_id").indices                      # trace_id -> positions in S (contiguous, text order)

# ---- phase transitions (merged runs); START / END mark the beginning and end of a trace -------------------------------
STATES_FROM, STATES_TO = ["START"] + ROLES, ROLES + ["END"]
e_in = R.assign(frm=R["prev_role"].fillna("START"), to=R["role"])[["trace_id", "frm", "to"]]
e_out = R[R["next_role"].isna()].assign(frm=lambda d: d["role"], to="END")[["trace_id", "frm", "to"]]
EDGES = pd.concat([e_in, e_out], ignore_index=True)
EDGES["qkey"] = EDGES["trace_id"].map(E["qkey"])
print("eligible samples: %d | segments: %d | features in the table: %d" % (nE, len(S), len(TABLE_FEATS)))

### Annotation consistency (answer fields)
How reliable are the answer annotations? The agreement between `is_correct` and "last annotated answer equals gold" is the key sanity check.

In [ ]:
chk = E[E["last_val"].notna()]
agree = pd.crosstab(chk["correct"].map({1.0: "is_correct=1", 0.0: "is_correct=0"}),
                    chk["last_matches_gold"].map({1.0: "last answer == gold", 0.0: "last answer != gold"}))
display(agree)
print("traces with an annotated answer value: %d of %d | agreement between is_correct and (last annotated answer == gold): %.3f" % (
    len(chk), len(E), (chk["correct"] == chk["last_matches_gold"]).mean()))
print("with a `final` segment: %.1f%% | `final` before any `candidate`: %d | first segment annotated `continuation`: %d" % (
    100 * E["first_final_seg"].notna().mean(), int((E["first_final_seg"] < E["first_cand_seg"]).sum()),
    int(S[(S["seg_pos"] == 0) & (S["temporal_role"] == "continuation")].shape[0])))
print("segment-level label counts:")
display(pd.concat({c: S[c].value_counts(dropna=False) for c in ["role", "answer_status", "temporal_role"]}).to_frame("segments"))

In [ ]:
def build_questions(Fe, min_samples=MIN_SAMPLES):
    """One row per question (all compute levels pooled): counts, length statistics and the group."""
    q = Fe.groupby("qkey", sort=False).agg(
        benchmark=("benchmark", "first"), question_id=("question_id", "first"), difficulty_tier=("difficulty_tier", "first"),
        n=("words", "size"), n_correct=("correct", "sum"), n_levels=("compute_level", "nunique"),
        w_min=("words", "min"), w_max=("words", "max"), w_mean=("words", "mean"), w_sd=("words", "std"))
    q["n_wrong"] = q["n"] - q["n_correct"]
    q["len_std"] = q["w_sd"] / q["w_mean"]                        # standardised std (coefficient of variation of the length)
    q["group"] = np.select([q["n"] < min_samples, q["n_wrong"] == 0, q["n_correct"] == 0],
                           ["TOO-FEW", "ALL-CORRECT", "ALL-WRONG"], default="MIXED")
    return q


questions = build_questions(E)
enough = questions["group"] != "TOO-FEW"
LEN_STD_THR = questions.loc[enough, "len_std"].quantile(LEN_STD_QUANTILE)
questions["len_sig"] = enough & (questions["len_std"] > LEN_STD_THR)
q_members = E.groupby("qkey").groups                              # qkey -> eligible trace ids
E["q_group"] = E["qkey"].map(questions["group"])

print("questions: %d | with >= %d eligible samples: %d | samples per question: median %d (min %d, max %d)" % (
    len(questions), MIN_SAMPLES, enough.sum(), questions["n"].median(), questions["n"].min(), questions["n"].max()))
print("standardised length std (std/mean) over questions: median %.3f | %.0f%% quantile = %.3f  -> significant length spread if above" % (
    questions.loc[enough, "len_std"].median(), 100 * LEN_STD_QUANTILE, LEN_STD_THR))
overview = questions.groupby("group").agg(questions=("n", "size"), samples=("n", "sum"), median_samples=("n", "median"),
                                          length_significant=("len_sig", "sum")).reindex(GROUPS + ["TOO-FEW"]).fillna(0).astype(int)
display(overview)
if questions["difficulty_tier"].notna().any():
    print("questions by difficulty tier:")
    display(questions[enough].groupby(["difficulty_tier", "group"]).size().unstack(fill_value=0).reindex(columns=GROUPS, fill_value=0))
save_table(questions.drop(columns=["w_sd"]), "question_groups.csv")


def representative(group, k=N_SAVE_PER_GROUP):
    """The k questions closest to the group median (standardised log mean length, log length ratio, accuracy)."""
    q = questions[questions["group"] == group]
    if q.empty:
        return []
    f = pd.DataFrame({"len": np.log(q["w_mean"]), "spread": np.log(q["w_max"] / q["w_min"]), "acc": q["n_correct"] / q["n"]})
    z = (f - f.median()) / f.std(ddof=0).replace(0, 1)
    return list((z ** 2).sum(axis=1).sort_values().index[:k])


REP = {g: representative(g) for g in GROUPS}
print("example questions that will be saved to disk:", REP)

### Helper: trajectory figure of one question (one panel per compute level)
* One **panel per compute level**; a panel contains only the eligible samples of that level, so the samples are compared with each other **inside the same compute level**.
* x = cumulative word position. By default every panel has its **own absolute scale** shared by the rows of that panel (`share_x=True` puts all panels on one scale).
* Each segment is a block whose **width = its word count**, drawn in the original text order, never sorted and never rescaled per trace.
* Colour = `discourse_role` (same in every figure). Rows are ordered by total words (**shortest at the top**). Row labels: sample `s`, outcome, length; the compute level is in the panel title.
* Markers sit at the *end* of the segment that first carries the event: first answer candidate, first final, observed answer replacement / refinement.


In [ ]:
MARKERS = {"cand": dict(marker="v", color="#08306b", label="first answer candidate"),
           "final": dict(marker="D", color="#6a3d9a", label="first final"),
           "repl": dict(marker="X", color="#d62728", label="observed answer replacement"),
           "refine": dict(marker="P", color="#17becf", label="answer refinement / format change")}


def plot_question(qk, path=None, show=True, dpi=FIG_DPI_SHOW, share_x=False):
    """One figure per question, ONE PANEL PER COMPUTE LEVEL. Inside a panel only the samples of that level are compared.
    share_x=False: every panel has its own x-scale (best for comparing samples inside a level).
    share_x=True : all panels share one x-scale (also lets you compare lengths across levels)."""
    rows_all = E.loc[q_members[qk]]
    levels = sorted(rows_all["compute_level"].unique(), key=lambda c: int(c))
    counts = [int((rows_all["compute_level"] == c).sum()) for c in levels]
    fig, axes = plt.subplots(len(levels), 1, figsize=(12, 0.42 * sum(counts) + 0.9 * len(levels) + 1.5), dpi=dpi,
                             sharex=share_x, squeeze=False,
                             gridspec_kw=dict(height_ratios=[c + 1.5 for c in counts], hspace=0.6))
    axes = axes[:, 0]
    used = set()
    global_max = float(rows_all["words"].max()) * 1.02

    for pi, (ax, lvl) in enumerate(zip(axes, levels)):
        rows = rows_all[rows_all["compute_level"] == lvl].sort_values(["words", "sample_idx"])
        n = len(rows)
        for k, (tid, r) in enumerate(rows.iterrows()):
            y = n - 1 - k                                                # shortest row at the top
            ix = seg_ix[tid]
            st, wd, rl = S_start[ix], S_words[ix], S_role[ix]
            assert wd.sum() == r["words"] and st[0] == 0 and np.all(np.diff(st) >= 0)
            pos = wd > 0
            ax.broken_barh(list(zip(st[pos], wd[pos])), (y - 0.32, 0.64), facecolors=[ROLE_COLORS[x] for x in rl[pos]],
                           edgecolors="white", linewidth=0.3)
            for s0, x in zip(st[~pos], rl[~pos]):
                ax.vlines(s0, y - 0.32, y + 0.32, colors=ROLE_COLORS[x], linewidth=0.8)
            # separator: every segment labelled `new` (except the first segment of the trace) starts a new thought -> black bar at its start
            tp = S_temp[ix]
            sep = np.isin(tp[1:], list(NEW_VALUES)) & (wd[1:] > 0)
            if sep.any():
                ax.vlines(st[1:][sep], y - 0.38, y + 0.38, colors=SEP_COLOR, linewidth=1.2, zorder=4)
                used.add("newsep")
            # dashed arrow: this phase repeats an earlier phase of the same role (only after a phase change)
            for xf, xt in REP_LINKS.get(tid, []):
                ax.plot([xf, xf, xt], [y + 0.34, y + 0.52, y + 0.52], ls="--", color=REP_COLOR, lw=0.9, zorder=6)
                ax.annotate("", xy=(xt, y + 0.33), xytext=(xt, y + 0.52), zorder=6,
                            arrowprops=dict(arrowstyle="-|>", color=REP_COLOR, lw=0.9, mutation_scale=7, shrinkA=0, shrinkB=0))
                ax.scatter([xf], [y + 0.34], s=7, color=REP_COLOR, zorder=6)
                used.add("rep")
            evs = []
            if pd.notna(r["first_cand_end"]):
                evs.append(("cand", r["first_cand_end"]))
            if pd.notna(r["first_final_end"]):
                evs.append(("final", r["first_final_end"]))
            if tid in A_lists.index:
                evs += [("repl", S_end[ix[p]]) for p in A_lists.at[tid, "replace_segs"]]
                evs += [("refine", S_end[ix[p]]) for p in A_lists.at[tid, "refine_segs"]]
            for kind, xw in evs:
                m = MARKERS[kind]
                ax.scatter([xw], [y], marker=m["marker"], s=26, color=m["color"], edgecolors="white", linewidths=0.6, zorder=5)
                used.add(kind)

        rev = rows.iloc[::-1]
        ax.set_yticks(range(n))
        ax.set_yticklabels(["s%s · %s · %dw" % ("?" if pd.isna(r["sample_idx"]) else int(r["sample_idx"]),
                                                "correct" if r["correct"] == 1 else "INCORRECT", r["words"])
                            for _, r in rev.iterrows()], fontsize=8)
        for lab, (_, r) in zip(ax.get_yticklabels(), rev.iterrows()):
            lab.set_color("#1b7837" if r["correct"] == 1 else "#b2182b")
        ax.set_xlim(0, global_max if share_x else float(rows["words"].max()) * 1.02)
        ax.set_ylim(-0.7, n - 0.3)
        ax.grid(axis="y", visible=False)
        ax.set_title("compute level %s · %d samples (%d correct)" % (lvl, n, int(rows["correct"].sum())),
                     fontsize=9.5, loc="left")
        if (not share_x) or pi == len(axes) - 1:
            ax.set_xlabel("cumulative word position" + (" (shared scale)" if share_x else " (scale of this panel)"), fontsize=8)

    qr = questions.loc[qk]
    fig.suptitle("%s · group %s · %d samples (%d correct) · std/mean of length %.2f%s" % (
        qk, qr["group"], qr["n"], qr["n_correct"], qr["len_std"], " (significant)" if qr["len_sig"] else ""),
        fontsize=11, y=0.995)
    handles = [plt.Rectangle((0, 0), 1, 1, color=ROLE_COLORS[x]) for x in ROLES]
    names = list(ROLES)
    for kind in ["cand", "final", "repl", "refine"]:
        if kind in used:
            handles.append(plt.Line2D([], [], marker=MARKERS[kind]["marker"], color=MARKERS[kind]["color"], ls=""))
            names.append(MARKERS[kind]["label"])
    if "newsep" in used:
        handles.append(plt.Line2D([], [], color=SEP_COLOR, marker="|", markersize=9, ls=""))
        names.append("start of a segment labelled new (new thought)")
    if "rep" in used:
        handles.append(plt.Line2D([], [], ls="--", color=REP_COLOR, marker=">", markersize=4))
        names.append("repetition of an earlier phase (arrow points to the source)")
    axes[-1].legend(handles, names, ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.45), frameon=False)
    finish_fig(fig, path=path, show=show)


def show_group_figures(group):
    """Display the trajectory figure of SHOW_FRAC of the questions of the group; save only the example questions."""
    qs = list(questions[questions["group"] == group].sort_values("len_std", ascending=False).index)
    n_show = int(np.ceil(SHOW_FRAC * len(qs)))
    if MAX_SHOWN_PER_GROUP is not None:
        n_show = min(n_show, MAX_SHOWN_PER_GROUP)
    # evenly spaced picks over the length-spread order (deterministic, covers the whole range)
    pick = np.unique(np.round(np.linspace(0, len(qs) - 1, n_show)).astype(int)) if n_show > 0 else []
    shown = {qs[i] for i in pick}
    saved = set(REP[group])
    for qk in qs:
        if qk in shown or qk in saved:
            path = os.path.join(EX_DIR, safe_name(group), safe_name(qk) + ".png") if qk in saved else None
            plot_question(qk, path=path, show=qk in shown)
    print("%s: %d questions | figures displayed: %d (%.0f%%) | saved to %s: %d" % (
        group, len(qs), len(shown), 100 * len(shown) / max(1, len(qs)), os.path.join(EX_DIR, safe_name(group)), len(saved)))

### Helper: within-question contrast machinery
For one comparison (long vs short, or incorrect vs correct) every question contributes **one row per side** (mean over the samples of that side).
The per-question difference *side A minus side B* is summarised across questions: mean with bootstrap CI, paired effect size `dz`
(mean / SD of the question-level differences), sign-flip and Wilcoxon p-values (exploratory, no multiplicity correction).
The side definitions are: **long / short** = longest / shortest `SIDE_FRAC` of the question's samples (at least one sample, never overlapping);
**incorrect / correct** = the incorrect / correct samples of a MIXED question.

In [ ]:
def side_dict(kind, qkeys):
    """qkey -> list of trace ids forming one side of the comparison."""
    out = {}
    for qk in qkeys:
        sub = E.loc[q_members[qk]]
        if kind in ("long", "short"):
            sub = sub.sort_values(["words", "sample_idx"])
            k = max(1, min(int(round(len(sub) * SIDE_FRAC)), len(sub) // 2))
            out[qk] = list(sub.index[-k:] if kind == "long" else sub.index[:k])
        else:
            out[qk] = list(sub.index[sub["correct"] == (1 if kind == "correct" else 0)])
    return out


def _flat(ids_by_q):
    ids = [i for v in ids_by_q.values() for i in v]
    qk = [q for q, v in ids_by_q.items() for _ in v]
    return ids, qk


def side_table(ids_by_q, feats=TABLE_FEATS):
    """Question-level mean of the features over the samples of one side (rows = questions)."""
    ids, qk = _flat(ids_by_q)
    t = E.loc[ids, feats].copy()
    t["qkey"] = qk
    return t.groupby("qkey", sort=False)[feats].mean()


def side_profile(ids_by_q):
    """Question-level mean of the role share inside each relative-position decile (rows = questions, cols = role x decile)."""
    ids, qk = _flat(ids_by_q)
    d = pd.DataFrame(REL_FLAT[E.index.get_indexer(ids)])
    d["qkey"] = qk
    return d.groupby("qkey", sort=False).mean()


def transition_rows(ids):
    """Per-question row-normalised phase-transition probabilities of the given traces: index (qkey, from), columns = to."""
    sel = EDGES[EDGES["trace_id"].isin(ids)]
    cnt = sel.groupby(["qkey", "frm", "to"]).size().unstack(fill_value=0).reindex(columns=STATES_TO, fill_value=0)
    return cnt.div(cnt.sum(axis=1), axis=0)


def transition_diff(ids_a, ids_b, B=300):
    """Mean over questions of (transition matrix A - B); '*' where the question-bootstrap CI excludes 0 (needs >= 5 questions)."""
    pa, pb = transition_rows(ids_a), transition_rows(ids_b)
    d = pa.loc[pa.index.intersection(pb.index)] - pb.loc[pa.index.intersection(pb.index)]
    mean = d.groupby(level="frm").mean().reindex(STATES_FROM)
    star = pd.DataFrame("", index=STATES_FROM, columns=STATES_TO)
    for frm in d.index.get_level_values("frm").unique():
        sub = d.xs(frm, level="frm")
        for to in STATES_TO:
            m, lo, hi, nq = boot_mean_ci(sub[to], B=B)
            if nq >= 5 and (lo > 0 or hi < 0):
                star.at[frm, to] = "*"
    return mean, star


def contrast(side_a, side_b):
    """Everything needed for one comparison A - B inside questions. Returns None when fewer than 3 questions have both sides."""
    qs = [q for q in side_a if len(side_a[q]) and len(side_b.get(q, []))]
    if len(qs) < 3:
        return None
    ia, ib = {q: side_a[q] for q in qs}, {q: side_b[q] for q in qs}
    fa, fb = side_table(ia).reindex(qs), side_table(ib).reindex(qs)
    pa, pb = side_profile(ia).reindex(qs).to_numpy(), side_profile(ib).reindex(qs).to_numpy()
    eff = paired_summary(fa - fb)
    eff.insert(0, "mean_A", fa.mean())
    eff.insert(1, "mean_B", fb.mean())
    eff["sig"] = (eff["ci_lo"] > 0) | (eff["ci_hi"] < 0)
    tm, ts = transition_diff([i for v in ia.values() for i in v], [i for v in ib.values() for i in v])
    return dict(qs=qs, fa=fa, fb=fb, pa=pa, pb=pb, pdiff=pa - pb, eff=eff, trans_mean=tm, trans_star=ts)


def show_effects(res, name, top=15):
    """Print the largest effects (by |dz|) and save the full effect table."""
    t = res["eff"].reindex(TABLE_FEATS)
    cols = ["n_questions", "mean_A", "mean_B", "mean", "ci_lo", "ci_hi", "dz", "p_signflip", "sig"]
    print("largest within-question differences A - B (|dz|), %d questions; sig = bootstrap CI of the mean difference excludes 0:" % len(res["qs"]))
    display(t.reindex(t["dz"].abs().sort_values(ascending=False).index).head(top)[cols].round(3))
    save_table(t[cols], name)


def plot_overall(res, title, na, nb, path=None):
    """ONE figure with the overall pattern of a comparison: composition along the trace, role-share differences, effects,
    phase transitions and answer development."""
    xr = (np.arange(REL_BINS) + 0.5) / REL_BINS * 100
    fig = plt.figure(figsize=(17, 9.5))
    gs = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.28)
    for k, (arr, nm) in enumerate([(res["pa"], na), (res["pb"], nb)]):
        ax = fig.add_subplot(gs[0, k])
        ax.stackplot(xr, np.nanmean(arr, axis=0).reshape(5, REL_BINS), colors=[ROLE_COLORS[r] for r in ROLES], labels=ROLES)
        ax.set_xlim(xr[0], xr[-1]); ax.set_ylim(0, 1)
        ax.set_title("(%s) role composition along the trace: %s" % ("ab"[k], nm), fontsize=10)
        ax.set_xlabel("relative position in the trace (%)")
        if k == 0:
            ax.set_ylabel("share of words")
            ax.legend(fontsize=7, ncol=2, loc="lower left")
    ax = fig.add_subplot(gs[0, 2])
    mean, lo, hi = boot_matrix_ci(res["pdiff"])
    for r in KEY_ROLES + ["reasoning"]:
        j = ROLES.index(r)
        sl = slice(j * REL_BINS, (j + 1) * REL_BINS)
        ax.plot(xr, mean[sl], color=ROLE_COLORS[r], marker="o", ms=3, label=r)
        ax.fill_between(xr, lo[sl], hi[sl], color=ROLE_COLORS[r], alpha=0.18)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel("relative position in the trace (%)")
    ax.set_ylabel("role share: %s minus %s" % (na, nb))
    ax.set_title("(c) where the roles differ (95%% question-bootstrap band, n=%d questions)" % len(res["qs"]), fontsize=10)
    ax.legend(fontsize=7, ncol=2)
    # (d) length-free effects with the largest |dz|
    ax = fig.add_subplot(gs[1, 0])
    eff = res["eff"].reindex(LEN_FREE).dropna(subset=["dz"])
    eff = eff.reindex(eff["dz"].abs().sort_values(ascending=False).index).head(12).sort_values("dz")
    y = np.arange(len(eff))
    sig = (eff["dz_lo"] > 0) | (eff["dz_hi"] < 0)
    col = np.where(sig, "#d95f02", "#9a9a9a")
    ax.hlines(y, eff["dz_lo"], eff["dz_hi"], color=col, lw=1.6)
    ax.scatter(eff["dz"], y, color=col, s=26, zorder=3)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(eff.index, fontsize=8)
    ax.set_xlabel("paired effect size dz of (%s - %s), question level" % (na, nb))
    ax.set_title("(d) 12 largest length-free differences (orange = CI excludes 0)", fontsize=10)
    # (e) phase transitions
    ax = fig.add_subplot(gs[1, 1])
    lab = res["trans_mean"].round(2).astype(str) + res["trans_star"]
    sns.heatmap(res["trans_mean"].astype(float), annot=lab, fmt="", cmap="RdBu_r", center=0, vmin=-0.3, vmax=0.3, cbar=False, ax=ax,
                annot_kws={"fontsize": 7})
    ax.set_xlabel("next phase"); ax.set_ylabel("current phase")
    ax.set_title("(e) phase-transition probability: %s minus %s (* = CI excludes 0)" % (na, nb), fontsize=9)
    # (f) answer development
    ax = fig.add_subplot(gs[1, 2])
    pat = pd.DataFrame({na: res["fa"][PAT_COLS].mean(), nb: res["fb"][PAT_COLS].mean()}).T
    pat.columns = list(PAT_LABELS.values())
    pat.plot(kind="barh", stacked=True, ax=ax, color=["#54a24b", "#f58518", "#d62728", "#bbbbbb"], width=0.6)
    ax.set_xlabel("share of samples (mean over questions)")
    ax.set_title("(f) answer development", fontsize=10)
    ax.legend(fontsize=6.5, loc="lower center", bbox_to_anchor=(0.5, -0.45), ncol=2)
    fig.suptitle(title, fontsize=12)
    finish_fig(fig, path=path, show=True)

## 6. ALL-CORRECT questions
Every eligible sample of the question is correct. **Question:** when the model is right every time, how do its longer samples differ from its
shorter samples? Only questions with a significant length spread (`std/mean` above the `LEN_STD_QUANTILE` quantile) enter the long-vs-short calculations.

### 6.1 Trajectory figure of every ALL-CORRECT question

In [ ]:
show_group_figures("ALL-CORRECT")

### 6.2 Long vs short samples inside the question: overall pattern

In [ ]:
qs_ac = list(questions.index[(questions["group"] == "ALL-CORRECT") & questions["len_sig"]])
print("ALL-CORRECT: %d questions, %d with a significant length spread (std/mean > %.3f) -> used for long vs short" % (
    (questions["group"] == "ALL-CORRECT").sum(), len(qs_ac), LEN_STD_THR))
res_ac = contrast(side_dict("long", qs_ac), side_dict("short", qs_ac))
if res_ac is None:
    print("fewer than 3 usable questions: lower LEN_STD_QUANTILE (e.g. 0.5) to include more questions")
else:
    lr = np.exp(res_ac["fa"]["log_words"] - res_ac["fb"]["log_words"])
    print("long / short length ratio (median over questions): %.2f" % lr.median())
    show_effects(res_ac, "ALL-CORRECT_long_vs_short.csv")
    plot_overall(res_ac, "ALL-CORRECT questions · long vs short samples inside the same question (n=%d questions)" % len(res_ac["qs"]),
                 "long", "short", path=os.path.join(FIG_DIR, "overall_ALL-CORRECT_long_vs_short.png"))

## 7. MIXED questions
At least one sample is correct and at least one is incorrect. **Question:** inside the same question, how do the incorrect samples differ from the
correct ones? No length filter here: the comparison is by outcome, not by length (the length difference itself is reported).

### 7.1 Trajectory figure of every MIXED question

In [ ]:
show_group_figures("MIXED")

### 7.2 Incorrect vs correct samples inside the question: overall pattern

In [ ]:
qs_mx = list(questions.index[questions["group"] == "MIXED"])
print("MIXED: %d questions" % len(qs_mx))
res_mx = contrast(side_dict("incorrect", qs_mx), side_dict("correct", qs_mx))
if res_mx is None:
    print("fewer than 3 MIXED questions")
else:
    d_len = res_mx["fa"]["log_words"] - res_mx["fb"]["log_words"]
    print("questions where the incorrect samples are longer on average: %.0f%% | median length ratio incorrect / correct: %.2f" % (
        100 * (d_len > 0).mean(), np.exp(d_len.median())))
    show_effects(res_mx, "MIXED_incorrect_vs_correct.csv")
    plot_overall(res_mx, "MIXED questions · incorrect vs correct samples inside the same question (n=%d questions)" % len(res_mx["qs"]),
                 "incorrect", "correct", path=os.path.join(FIG_DIR, "overall_MIXED_incorrect_vs_correct.png"))

## 8. ALL-WRONG questions
Every eligible sample of the question is incorrect. **Question:** do the longer wrong samples differ from the shorter wrong samples? Same length-significance
rule as in section 6.

### 8.1 Trajectory figure of every ALL-WRONG question

In [ ]:
show_group_figures("ALL-WRONG")

### 8.2 Long vs short samples inside the question: overall pattern

In [ ]:
qs_aw = list(questions.index[(questions["group"] == "ALL-WRONG") & questions["len_sig"]])
print("ALL-WRONG: %d questions, %d with a significant length spread (std/mean > %.3f) -> used for long vs short" % (
    (questions["group"] == "ALL-WRONG").sum(), len(qs_aw), LEN_STD_THR))
res_aw = contrast(side_dict("long", qs_aw), side_dict("short", qs_aw))
if res_aw is None:
    print("fewer than 3 usable questions: lower LEN_STD_QUANTILE (e.g. 0.5) to include more questions")
else:
    lr = np.exp(res_aw["fa"]["log_words"] - res_aw["fb"]["log_words"])
    print("long / short length ratio (median over questions): %.2f" % lr.median())
    show_effects(res_aw, "ALL-WRONG_long_vs_short.csv")
    plot_overall(res_aw, "ALL-WRONG questions · long vs short samples inside the same question (n=%d questions)" % len(res_aw["qs"]),
                 "long", "short", path=os.path.join(FIG_DIR, "overall_ALL-WRONG_long_vs_short.png"))

## 9. Group comparison
Five conditions side by side (question-level means over the samples of each condition):
**ALL-CORRECT short**, **ALL-CORRECT long** (shortest / longest `SIDE_FRAC` of each question's samples), **MIXED correct**, **MIXED incorrect**
and **ALL-WRONG** (all samples of the question). With `CMP_LEN_SIG_ONLY = True` the ALL-CORRECT and ALL-WRONG questions are restricted to those with a
significant length spread (`std/mean` above the `LEN_STD_QUANTILE` quantile), exactly as in sections 6.2 and 8.2; MIXED is compared by outcome and is never length-filtered.

**Caveat:** ALL-CORRECT, MIXED and ALL-WRONG contain *different questions* (typically of different difficulty), so differences between conditions are not
within-question effects: they show where the groups sit relative to each other, not what causes an outcome. Only ALL-CORRECT short vs long is a within-question
contrast. The within-question contrasts above are the ones to read for that.


In [ ]:
mixed_qs = list(questions.index[questions["group"] == "MIXED"])
ac_mask = (questions["group"] == "ALL-CORRECT") & (questions["len_sig"] if CMP_LEN_SIG_ONLY else True)
aw_mask = (questions["group"] == "ALL-WRONG") & (questions["len_sig"] if CMP_LEN_SIG_ONLY else True)
ac_qs, aw_qs = list(questions.index[ac_mask]), list(questions.index[aw_mask])

COND = {"ALL-CORRECT short": side_dict("short", ac_qs),
        "ALL-CORRECT long": side_dict("long", ac_qs),
        "MIXED correct": side_dict("correct", mixed_qs),
        "MIXED incorrect": side_dict("incorrect", mixed_qs),
        "ALL-WRONG": {q: list(q_members[q]) for q in aw_qs}}
COND = {k: v for k, v in COND.items() if len(v)}
tabs = {k: side_table(v) for k, v in COND.items()}
profs = {k: side_profile(v).to_numpy() for k, v in COND.items()}

KEY_FEATS = ["share_doubt", "share_verification", "share_correction", "share_reasoning", "changes_per100w", "first_ans_rel",
             "share_after_first_ans", "n_replace_100w", "rep4_share", "hedge_100w", "log_words"]
rows = []
for k, t in tabs.items():
    for f in KEY_FEATS:
        m, lo, hi, nq = boot_mean_ci(t[f])
        rows.append(dict(condition=k, feature=f, mean=m, ci_lo=lo, ci_hi=hi, n_questions=nq))
cmp_tab = pd.DataFrame(rows)
display(cmp_tab.pivot(index="feature", columns="condition", values="mean").reindex(KEY_FEATS)[list(COND)].round(3))
save_table(cmp_tab, "group_comparison.csv", index=False)

xr = (np.arange(REL_BINS) + 0.5) / REL_BINS * 100
fig = plt.figure(figsize=(21, 9.5))
gs = fig.add_gridspec(2, len(COND), hspace=0.45, wspace=0.25)
for k, (name, arr) in enumerate(profs.items()):
    ax = fig.add_subplot(gs[0, k])
    ax.stackplot(xr, np.nanmean(arr, axis=0).reshape(5, REL_BINS), colors=[ROLE_COLORS[r] for r in ROLES], labels=ROLES)
    ax.set_xlim(xr[0], xr[-1]); ax.set_ylim(0, 1)
    ax.set_title("%s\n(n=%d questions)" % (name, arr.shape[0]), fontsize=10, color=COND_COLORS[name])
    ax.set_xlabel("relative position in the trace (%)")
    if k == 0:
        ax.set_ylabel("share of words")
        ax.legend(fontsize=7, ncol=2, loc="lower left")

half = max(1, len(COND) // 2)
ax = fig.add_subplot(gs[1, :half])
w = 0.8 / len(COND)
for i, (name, t) in enumerate(tabs.items()):
    ms = [boot_mean_ci(t["share_" + r]) for r in ROLES]
    ax.bar(np.arange(5) + i * w, [m[0] for m in ms], width=w, color=COND_COLORS[name], label=name,
           yerr=[[max(0, m[0] - m[1]) if m[1] == m[1] else 0 for m in ms],
                 [max(0, m[2] - m[0]) if m[2] == m[2] else 0 for m in ms]], capsize=2)
ax.set_xticks(np.arange(5) + w * (len(COND) - 1) / 2)
ax.set_xticklabels(ROLES)
ax.set_ylabel("share of words (mean over questions, 95% CI)")
ax.set_title("role shares by condition", fontsize=10)
ax.legend(fontsize=7)

ax = fig.add_subplot(gs[1, half:])
pat = pd.DataFrame({k: t[PAT_COLS].mean() for k, t in tabs.items()}).T
pat.columns = list(PAT_LABELS.values())
pat.plot(kind="barh", stacked=True, ax=ax, color=["#54a24b", "#f58518", "#d62728", "#bbbbbb"], width=0.65)
ax.invert_yaxis()
ax.set_xlabel("share of samples (mean over questions)")
ax.set_title("answer development by condition", fontsize=10)
ax.legend(fontsize=7, loc="lower center", bbox_to_anchor=(0.5, -0.42), ncol=2)

fig.suptitle("Group comparison · question-level means (different questions per group, see caveat) · ALL-CORRECT / ALL-WRONG %s" % (
    "restricted to questions with a significant length spread (std/mean > %.2f)" % LEN_STD_THR if CMP_LEN_SIG_ONLY else "use all questions"), fontsize=12)
finish_fig(fig, path=os.path.join(FIG_DIR, "overall_group_comparison.png"), show=True)

## 10. Saved files
Only the overall figures, `N_SAVE_PER_GROUP` example questions per group and a handful of small tables are written to disk.

In [ ]:
for root, _, files in sorted(os.walk(OUT_DIR)):
    for f in sorted(files):
        p = os.path.join(root, f)
        print("%8.0f KB  %s" % (os.path.getsize(p) / 1024, os.path.relpath(p, OUT_DIR)))